# CAD-Gym — Gamified RL Training (Colab)

Train a language model to design mechanical brackets and watch it improve in real-time.

**What you'll see:**
- Live 3D rotating view of each generated bracket
- Learning curves updating every 10 episodes
- Achievement unlocks as the model improves
- Champion bracket at the end

**Runtime:** GPU optional. TinyLlama runs on Colab T4 or CPU.

---

## 1. Install

In [ ]:
!pip install -q cadquery trimesh gymnasium pydantic
!pip install -q plotly transformers torch accelerate tqdm
!pip install -q git+https://github.com/prodata-ai/ProdataGym.git

import plotly.io as pio
pio.renderers.default = "colab"

print("Installation complete")

## 2. Load environment + model

In [ ]:
import gymnasium as gym
import prodata.cad_gym
from prodata.cad_gym.viz import TrainingVisualizer

import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch.optim import AdamW
from tqdm.notebook import tqdm

env = gym.make("prodata/BracketDesign-v0")
print(f"Environment: {len(env.unwrapped.task_ids())} tasks loaded")

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# Swap to Qwen/Qwen2.5-Coder-7B-Instruct for much better results (scores G > 0)

print(f"Loading {MODEL_NAME}...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# float16 for 7B+ models — fits in L4/A100 GPU; float32 for small models (RL stability)
_BIG_MODELS = ("qwen", "llama-3", "llama3", "mistral", "deepseek", "codellama", "phi-4")
_train_dtype = (
    torch.float16
    if torch.cuda.is_available() and any(x in MODEL_NAME.lower() for x in _BIG_MODELS)
    else torch.float32
)
print(f"dtype: {_train_dtype}")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=_train_dtype,
    device_map="auto",
)

# Gradient checkpointing: recomputes activations during backward instead of storing
# them all — cuts activation memory ~60%, essential for 7B training on L4 (24GB).
model.gradient_checkpointing_enable()

# LR 5e-6 for REINFORCE stability
optimizer = AdamW(model.parameters(), lr=5e-6)
print(f"Model loaded ({sum(p.numel() for p in model.parameters())//1e6:.0f}M params)")


## 3. Zero-shot baseline

Run 5 tasks before any training.

> **Reward may be 0.0 or low here** — TinyLlama rarely generates valid CadQuery on the first try.
> The error message next to each FAIL shows exactly why the sim rejected it.
> Any non-zero reward (even 0.3) means valid geometry was produced. RL training improves from here.

In [ ]:
import re

_CAPABLE_MODELS = ("qwen", "llama-3", "llama3", "mistral", "deepseek", "codellama", "phi-4")
_IS_CAPABLE = any(x in MODEL_NAME.lower() for x in _CAPABLE_MODELS)
print(f"Model mode: {'DIMENSION REASONER' if _IS_CAPABLE else 'BOX-ONLY'}")


# ── L-bracket geometry ────────────────────────────────────────────────────────
# Flat plate in XY, thickness in Z. Two arms share a corner at (0, 0).
#
#   Y
#   ^  |<arm_w>|
#   |  +-------+
#   |  |  wall |  wall_length (Y)
#   |  |  arm  |
#   |  +-------+--------+
#   |  | extension arm  |  arm_width (Y)
#   |  +----------------+
#   +----------------------------> X
#      <-------- ext_length ------>   (= task extension_mm)
#
# BBox extents: [ext_length, wall_length, plate_thickness]
# → must all fit within task's max_bounding_box_mm

def _build_lbracket(plate_t: int, arm_w: int, wall_l: int, ext_l: int) -> str:
    plate_t = max(3,  min(plate_t, 30))
    arm_w   = max(10, min(arm_w,   150))
    wall_l  = max(20, min(wall_l,  300))
    ext_l   = max(20, min(ext_l,   400))
    # Both arms share bottom-left corner at (0, 0).
    # wall_arm: x 0→arm_w, y 0→wall_l  → CadQuery center (arm_w/2, wall_l/2)
    # ext_arm:  x 0→ext_l,  y 0→arm_w  → CadQuery center (ext_l/2,  arm_w/2)
    return (
        f"import cadquery as cq\n"
        f"wall_arm = cq.Workplane('XY').box({arm_w}, {wall_l}, {plate_t})"
        f".translate(({arm_w/2:.1f}, {wall_l/2:.1f}, 0))\n"
        f"ext_arm  = cq.Workplane('XY').box({ext_l}, {arm_w}, {plate_t})"
        f".translate(({ext_l/2:.1f}, {arm_w/2:.1f}, 0))\n"
        f"result   = wall_arm.union(ext_arm)"
    )


# ── Prompt ────────────────────────────────────────────────────────────────────
# Qwen outputs 3 integers; ext_length comes directly from the task (extension_mm).
# This guarantees the extension arm always matches the task requirement.

_DIM_SYSTEM = """\
You are a mechanical engineer sizing an L-bracket (flat plate, L-shaped in XY).
Output EXACTLY 3 lines, integer mm values only. No other text.

plate_thickness: <plate Z thickness, e.g. 10>
arm_width: <width of each arm perpendicular to its length, e.g. 50>
wall_length: <length of the wall-mounting arm in Y, e.g. 90>

The extension arm length is fixed by the task. Thicker plate = stronger. Keep cost in mind.\
"""

_BOX_SYSTEM = """\
Output ONLY three box dimensions in mm.
Format: LENGTH, WIDTH, HEIGHT)
Example: 120, 80, 15)
No other text.\
"""


def build_prompt(obs: dict) -> str:
    system = _DIM_SYSTEM if _IS_CAPABLE else _BOX_SYSTEM
    messages = [
        {"role": "system", "content": system},
        {
            "role": "user",
            "content": (
                f"{obs['task_description']}\n\n"
                f"Load: {obs['load_kg'][0]:.0f} kg | "
                f"Extension arm: {obs['extension_mm'][0]:.0f} mm | "
                f"Budget: ${obs['max_cost_usd'][0]:.0f}"
            ),
        },
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    if _IS_CAPABLE:
        prompt += "plate_thickness: "
    else:
        prompt += '```python\nimport cadquery as cq\nresult = cq.Workplane("XY").box('
    return prompt


# ── Extraction ────────────────────────────────────────────────────────────────

def _find_close_paren(s: str) -> int:
    depth = 1
    for i, c in enumerate(s):
        if c == '(':   depth += 1
        elif c == ')':
            depth -= 1
            if depth == 0: return i
    return len(s)


def extract_cadquery(raw: str, extension_mm: int = 100) -> str:
    if "```" in raw:
        raw = raw.split("```")[0]
    raw = raw.strip()

    if _IS_CAPABLE:
        full = "plate_thickness: " + raw   # restore the pre-filled key

        def _parse(key: str, default: int) -> int:
            m = re.search(rf'{key}:\s*(\d+)', full, re.IGNORECASE)
            return int(m.group(1)) if m else default

        plate_t = _parse("plate_thickness", 10)
        arm_w   = _parse("arm_width",       50)
        wall_l  = _parse("wall_length",     90)
        return _build_lbracket(plate_t, arm_w, wall_l, extension_mm)

    else:
        raw = raw.replace("cq.mm", "1")
        close_idx = _find_close_paren(raw)
        inside = raw[:close_idx]
        args = [a.strip() for a in inside.split(",")]
        dims = ", ".join(args[:3])
        return f'import cadquery as cq\nresult = cq.Workplane("XY").box({dims})'


# ── Generation ────────────────────────────────────────────────────────────────

@torch.no_grad()
def generate_code(obs: dict, temperature: float = 0.8) -> str:
    prompt = build_prompt(obs)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        temperature=temperature,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )
    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
    ext_l = int(obs["extension_mm"][0])
    code = extract_cadquery(raw, extension_mm=ext_l)
    dims_line = [ln for ln in code.split("\n") if "box" in ln]
    print(f"  RAW: {repr(raw[:60])}  →  {dims_line[0][:80] if dims_line else '?'}")
    return code


# ── Visualisation helper ───────────────────────────────────────────────────────

def show_bracket(stl_path: str, title: str, color: str = "steelblue"):
    import trimesh
    import plotly.graph_objects as go
    mesh = trimesh.load(stl_path).subdivide()
    v, f = mesh.vertices, mesh.faces
    fig = go.Figure()
    fig.add_trace(go.Mesh3d(
        x=v[:,0], y=v[:,1], z=v[:,2],
        i=f[:,0], j=f[:,1], k=f[:,2],
        color=color, opacity=0.25, flatshading=True,
        lighting=dict(ambient=1.0, diffuse=0.1, specular=0.0),
        showscale=False,
    ))
    edges = mesh.edges_unique
    xe, ye, ze = [], [], []
    for e in edges:
        xe += [v[e[0],0], v[e[1],0], None]
        ye += [v[e[0],1], v[e[1],1], None]
        ze += [v[e[0],2], v[e[1],2], None]
    fig.add_trace(go.Scatter3d(
        x=xe, y=ye, z=ze, mode="lines",
        line=dict(color="steelblue", width=1), hoverinfo="skip",
    ))
    fig.update_layout(
        title=title, showlegend=False,
        scene=dict(aspectmode="data"),
        width=640, height=460,
        margin=dict(l=0, r=0, t=50, b=0),
    )
    fig.show()


# ── Baseline run ───────────────────────────────────────────────────────────────

print("\nZero-shot baseline (5 tasks)")
print("-" * 50)

baseline_tasks = env.unwrapped.task_ids()[:5]
baseline_results = []

for tid in baseline_tasks:
    obs, _ = env.reset(options={"task_id": tid})
    code = generate_code(obs)
    _, reward, _, _, info = env.step(code)
    baseline_results.append(info["success"])
    dim = info["dimension_scores"]
    status = "PASS" if info["success"] else "FAIL"
    err = info.get("error") or (info.get("warnings") or [""])[-1]
    print(f"  {tid}  {status}  reward={reward:.2f}  "
          f"S={dim.get('structural',0):.2f} "
          f"C={dim.get('cost',0):.2f} "
          f"G={dim.get('geometry',0):.2f}"
          + (f"  [{err[:60]}]" if reward <= 0 else ""))

print(f"\nBaseline: {sum(baseline_results)}/5 passed")
print("Starting training...")


## 4. RL Training with live dashboard

Dashboard updates every 10 episodes. Rotate 3D models with your mouse.

In [ ]:
N_EPISODES = 200    # Increase to 500+ for meaningful improvement
VIZ_EVERY  = 10

viz = TrainingVisualizer(
    target_pass_rate=0.50,
    rolling_window=30,
)

model.train()
print(f"Training {N_EPISODES} episodes | dashboard every {VIZ_EVERY}\n")

pbar = tqdm(range(1, N_EPISODES + 1), desc="Episode 1")

for episode in pbar:

    obs, info = env.reset()
    prompt    = build_prompt(obs)
    inputs    = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)

    # ── Generate ──────────────────────────────────────────────────────────────
    try:
        with torch.no_grad():
            gen_out = model.generate(
                **inputs, max_new_tokens=40, temperature=0.8,
                do_sample=True, pad_token_id=tokenizer.pad_token_id,
            )
    except Exception as e:
        pbar.set_description(f"Episode {episode} — generate error")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        optimizer.zero_grad()
        continue

    new_tokens = gen_out[0][inputs["input_ids"].shape[-1]:]
    raw        = tokenizer.decode(new_tokens, skip_special_tokens=True)
    ext_l      = int(obs["extension_mm"][0])
    code       = extract_cadquery(raw, extension_mm=ext_l)

    obs, reward, terminated, truncated, info = env.step(code)

    # Update progress bar with current episode stats
    pbar.set_description(f"Episode {episode}/{N_EPISODES}")
    pbar.set_postfix(reward=f"{reward:.2f}", S=f"{info['dimension_scores'].get('structural',0):.2f}",
                     C=f"{info['dimension_scores'].get('cost',0):.2f}",
                     G=f"{info['dimension_scores'].get('geometry',0):.2f}")

    if reward <= 0 and episode % 20 == 0:
        err = info.get("error") or (info.get("warnings") or ["(no detail)"])[0]
        print(f"  ep {episode}: FAIL — {err[:100]}")

    newly_unlocked = viz.update(episode, obs, reward, info, code)

    shifted_reward = reward   # REINFORCE baseline = 0.0

    # ── Update ────────────────────────────────────────────────────────────────
    optimizer.zero_grad()
    if shifted_reward > 0.01:
        with torch.enable_grad():
            full_ids = gen_out[0].unsqueeze(0)
            labels   = full_ids.clone()
            labels[0, :inputs["input_ids"].shape[-1]] = -100
            out  = model(input_ids=full_ids, labels=labels)
            loss = out.loss * shifted_reward
            loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        if torch.isfinite(grad_norm):
            optimizer.step()
        else:
            optimizer.zero_grad()
            if episode % 10 == 0:
                print(f"  ep {episode}: NaN gradient, skipping")

    # Free activation memory between episodes
    if episode % 10 == 0 and torch.cuda.is_available():
        torch.cuda.empty_cache()

    if episode % VIZ_EVERY == 0 or newly_unlocked:
        viz.render(episode)

print("Training complete")


## 5. Final summary + champion design

In [ ]:
viz.final_summary()

## 6. Post-training evaluation on all 50 tasks

In [ ]:
model.eval()
task_ids = env.unwrapped.task_ids()
results  = {}

print("Evaluating on all 50 tasks...")
for tid in tqdm(task_ids, desc="Eval"):
    obs, _ = env.reset(options={"task_id": tid})
    code   = generate_code(obs, temperature=0.1)
    _, reward, _, _, info = env.step(code)
    results[tid] = {
        "passed": info["success"],
        "reward": round(reward, 4),
        "dimension_scores": info["dimension_scores"],
    }

import json
from pathlib import Path
tasks_path = Path(prodata.cad_gym.__file__).parent / "tasks" / "brackets_basic.json"
with open(tasks_path) as f:
    tasks_meta = {t["task_id"]: t for t in json.load(f)}

print(f"\n{'='*60}")
print(f"Post-RL Evaluation -- {MODEL_NAME}")
print(f"{'='*60}")
for level in ("easy", "medium", "hard"):
    ids    = [tid for tid, t in tasks_meta.items() if t["difficulty"] == level]
    passed = sum(results[tid]["passed"] for tid in ids if tid in results)
    avg_r  = np.mean([results[tid]["reward"] for tid in ids if tid in results])
    print(f"  {level.upper():8s}: {passed}/{len(ids)} passed  avg reward {avg_r:.3f}")

total_pass = sum(r["passed"] for r in results.values())
total_avg  = np.mean([r["reward"] for r in results.values()])
print(f"{'='*60}")
print(f"  TOTAL: {total_pass}/{len(results)} = {total_pass/len(results):.1%}  avg reward {total_avg:.3f}")
print(f"  Improvement: {total_pass/50 - sum(baseline_results)/5:+.1%} vs zero-shot")

## 7. Before vs After -- side-by-side design comparison

In [ ]:
import trimesh
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def wireframe_traces(stl_path: str, surface_color: str, wire_color: str):
    mesh = trimesh.load(stl_path).subdivide()
    v, f = mesh.vertices, mesh.faces
    surface = go.Mesh3d(
        x=v[:,0], y=v[:,1], z=v[:,2],
        i=f[:,0], j=f[:,1], k=f[:,2],
        color=surface_color, opacity=0.25, flatshading=True,
        lighting=dict(ambient=1.0, diffuse=0.1, specular=0.0),
        showscale=False,
    )
    edges = mesh.edges_unique
    xe, ye, ze = [], [], []
    for e in edges:
        xe += [v[e[0],0], v[e[1],0], None]
        ye += [v[e[0],1], v[e[1],1], None]
        ze += [v[e[0],2], v[e[1],2], None]
    wireframe = go.Scatter3d(
        x=xe, y=ye, z=ze, mode="lines",
        line=dict(color=wire_color, width=1), hoverinfo="skip",
    )
    return surface, wireframe


best_task_id = viz.best["task_id"]
best_mesh    = viz.best["mesh_file"]

if best_task_id and best_mesh:
    obs, _ = env.reset(options={"task_id": best_task_id})
    zs_code = generate_code(obs, temperature=0.01)
    _, zs_reward, _, _, zs_info = env.step(zs_code)
    zs_mesh = zs_info.get("mesh_file")

    if zs_mesh:
        fig = make_subplots(
            rows=1, cols=2,
            specs=[[{"type": "scene"}, {"type": "scene"}]],
            subplot_titles=(
                f"Zero-shot  reward={zs_reward:.2f}",
                f"Best trained  reward={viz.best['reward']:.2f}",
            ),
        )
        s1, w1 = wireframe_traces(zs_mesh,  "salmon",    "firebrick")
        s2, w2 = wireframe_traces(best_mesh, "lightblue", "steelblue")
        fig.add_trace(s1, row=1, col=1)
        fig.add_trace(w1, row=1, col=1)
        fig.add_trace(s2, row=1, col=2)
        fig.add_trace(w2, row=1, col=2)
        fig.update_layout(
            title_text=f"Task {best_task_id} -- Before vs After",
            height=500, width=960, showlegend=False,
            scene=dict(aspectmode="data"),
            scene2=dict(aspectmode="data"),
        )
        fig.show()
    else:
        print("Zero-shot crashed -- showing best design only.")
        show_bracket(best_mesh, f"Best: {best_task_id}  reward={viz.best['reward']:.2f}")
else:
    print("No passing design yet. Run more episodes.")

## Next steps

- **More episodes** -- TinyLlama needs 300-500 to show meaningful improvement.
- **Bigger model** -- `Qwen/Qwen2.5-Coder-7B-Instruct` gets much better results.
- **Pro verifier** -- `verifier_mode="pro"` to avoid reward hacking. [prodata.ai](https://prodata.ai)
- **Leaderboard** -- Open a PR at [github.com/prodata-ai/ProdataGym](https://github.com/prodata-ai/ProdataGym)